In [8]:
from google.colab import files

uploaded = files.upload()

Saving rahul_transactions.csv.zip to rahul_transactions.csv.zip


In [9]:
# ============================================================
# FEATURE 1 - TRANSACTION PARSER
# ============================================================

import pandas as pd
import numpy as np
import zipfile
import os

# ------------------------------------------------------------
# 1. Find the uploaded ZIP file
# ------------------------------------------------------------

zip_files = [
    file for file in os.listdir("/content")
    if file.lower().endswith(".zip")
]

if len(zip_files) == 0:
    raise FileNotFoundError(
        "No ZIP file found. Please upload rahul_transactions.csv.zip first."
    )

zip_path = os.path.join("/content", zip_files[0])

print("ZIP file found:", zip_path)


# ------------------------------------------------------------
# 2. Extract ZIP file
# ------------------------------------------------------------

extract_folder = "/content/rahul_data"

os.makedirs(extract_folder, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_folder)

print("ZIP file extracted successfully.")


# ------------------------------------------------------------
# 3. Find CSV file
# ------------------------------------------------------------

csv_files = []

for root, folders, files_list in os.walk(extract_folder):
    for file in files_list:
        if file.lower().endswith(".csv"):
            csv_files.append(
                os.path.join(root, file)
            )

if len(csv_files) == 0:
    raise FileNotFoundError(
        "No CSV file found inside the ZIP file."
    )

csv_path = csv_files[0]

print("CSV file found:", csv_path)


# ------------------------------------------------------------
# 4. Read CSV
# ------------------------------------------------------------

df = pd.read_csv(csv_path)

print("\nOriginal dataset shape:", df.shape)


# ------------------------------------------------------------
# 5. Store original row count
# ------------------------------------------------------------

original_rows = len(df)


# ------------------------------------------------------------
# 6. Parse Date
# ------------------------------------------------------------

df["date"] = pd.to_datetime(
    df["Date"],
    errors="coerce",
    dayfirst=True
)


# ------------------------------------------------------------
# 7. Clean Amount
# ------------------------------------------------------------

df["amount"] = (
    df["Amount"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("Rs.", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df["amount"] = pd.to_numeric(
    df["amount"],
    errors="coerce"
)


# ------------------------------------------------------------
# 8. Standardise Type
# ------------------------------------------------------------

df["type"] = (
    df["Type"]
    .astype(str)
    .str.lower()
    .str.strip()
)

df["type"] = df["type"].replace({
    "dr": "debit",
    "debit": "debit",
    "cr": "credit",
    "credit": "credit"
})


# ------------------------------------------------------------
# 9. Empty Mode values → missing
# ------------------------------------------------------------

df["Mode"] = df["Mode"].replace("", np.nan)


# ------------------------------------------------------------
# 10. Count invalid values
# ------------------------------------------------------------

invalid_dates = df["date"].isna().sum()

invalid_amounts = df["amount"].isna().sum()


# ------------------------------------------------------------
# 11. Remove duplicates
# ------------------------------------------------------------

before_duplicates = len(df)

df = df.drop_duplicates()

after_duplicates = len(df)

duplicates_removed = (
    before_duplicates - after_duplicates
)


# ------------------------------------------------------------
# 12. Remove invalid dates and amounts
# ------------------------------------------------------------

df = df.dropna(
    subset=["date", "amount"]
)


# ------------------------------------------------------------
# 13. Reset index
# ------------------------------------------------------------

df = df.reset_index(drop=True)


# ------------------------------------------------------------
# 14. Create cleaned DataFrame
# ------------------------------------------------------------

df_clean = df.copy()


# ------------------------------------------------------------
# 15. Final report
# ------------------------------------------------------------

print()
print("=" * 60)
print("FEATURE 1 - TRANSACTION PARSER")
print("=" * 60)

print(
    f"Original transactions : {original_rows}"
)

print(
    f"Duplicates removed    : {duplicates_removed}"
)

print(
    f"Invalid dates         : {invalid_dates}"
)

print(
    f"Invalid amounts       : {invalid_amounts}"
)

print(
    f"Clean transactions    : {len(df_clean)}"
)

print(
    f"Months covered        : "
    f"{df_clean['date'].dt.to_period('M').nunique()}"
)

print("=" * 60)


# ------------------------------------------------------------
# 16. Data types
# ------------------------------------------------------------

print("\nData Types:")
print(df_clean.dtypes)


# ------------------------------------------------------------
# 17. First 5 rows
# ------------------------------------------------------------

print("\nFirst 5 cleaned transactions:")

print(
    df_clean[
        [
            "Date",
            "Time",
            "Description",
            "Type",
            "Amount",
            "date",
            "amount",
            "type"
        ]
    ].head()
)


# ------------------------------------------------------------
# 18. Final shape
# ------------------------------------------------------------

print(
    "\nFinal DataFrame shape:",
    df_clean.shape
)

ZIP file found: /content/rahul_transactions.csv.zip
ZIP file extracted successfully.
CSV file found: /content/rahul_data/DADS MP2 Dataset.csv

Original dataset shape: (1328, 8)

FEATURE 1 - TRANSACTION PARSER
Original transactions : 1328
Duplicates removed    : 18
Invalid dates         : 1184
Invalid amounts       : 0
Clean transactions    : 143
Months covered        : 12

Data Types:
Date                   object
Time                   object
Description            object
Type                   object
Amount                 object
Balance               float64
Mode                   object
Ref                    object
date           datetime64[ns]
amount                float64
type                   object
dtype: object

First 5 cleaned transactions:
         Date   Time             Description   Type   Amount       date  \
0  2024-01-01  03:11      AMAZON SELLER SVCS  Debit    ₹2462 2024-01-01   
1  2024-01-01  14:07    UPI-AMAN-8934@OKAXIS  Debit    ₹1828 2024-01-01   
2  2024-01-0

In [10]:
# ============================================================
# FEATURE 2 - VENDOR EXTRACTOR
# ============================================================

# ------------------------------------------------------------
# 1. Inspect unique descriptions
# ------------------------------------------------------------

print("=" * 60)
print("FEATURE 2 - VENDOR EXTRACTOR")
print("=" * 60)

print("\nUnique transaction descriptions:")

for description in df_clean["Description"].unique():
    print(description)


# ------------------------------------------------------------
# 2. Vendor keyword dictionary
# ------------------------------------------------------------

vendor_keywords = {

    "Swiggy": [
        "SWIGGY",
        "BUNDL"
    ],

    "Zomato": [
        "ZOMATO"
    ],

    "Amazon": [
        "AMAZON",
        "AMZN"
    ],

    "Zepto": [
        "ZEPTO"
    ],

    "Blinkit": [
        "BLINKIT",
        "GROFERS"
    ],

    "BigBasket": [
        "BIGBASKET"
    ],

    "Myntra": [
        "MYNTRA"
    ],

    "Flipkart": [
        "FLIPKART"
    ],

    "Ajio": [
        "AJIO"
    ],

    "Uber": [
        "UBER"
    ],

    "Ola": [
        "OLA"
    ],

    "Rapido": [
        "RAPIDO"
    ],

    "Google Pay": [
        "GOOGLEPAY",
        "GOOGLE PAY",
        "GPAY"
    ],

    "Netflix": [
        "NETFLIX"
    ],

    "Spotify": [
        "SPOTIFY"
    ],

    "YouTube": [
        "YOUTUBE"
    ],

    "Amazon Prime": [
        "PRIME"
    ],

    "Hotstar": [
        "HOTSTAR"
    ],

    "BookMyShow": [
        "BOOKMYSHOW"
    ],

    "PVR": [
        "PVR"
    ],

    "INOX": [
        "INOX"
    ],

    "Zerodha": [
        "ZERODHA",
        "COIN"
    ],

    "HDFC": [
        "HDFC"
    ],

    "ICICI": [
        "ICICI"
    ],

    "SBI": [
        "SBI"
    ],

    "HP": [
        "HPCL",
        "HP PETROL"
    ],

    "Indian Oil": [
        "INDIAN OIL",
        "IOCL"
    ],

    "Bharat Petroleum": [
        "BHARAT PETROLEUM",
        "BPCL"
    ],

    "Cafe Coffee Day": [
        "CCD",
        "CAFE COFFEE DAY"
    ],

    "Starbucks": [
        "STARBUCKS"
    ]
}


# ------------------------------------------------------------
# 3. Vendor extraction function
# ------------------------------------------------------------

def extract_vendor(description):

    # Convert description to uppercase
    text = str(description).upper().strip()


    # --------------------------------------------------------
    # ATM withdrawals
    # --------------------------------------------------------

    if "ATM" in text or "WDL" in text:
        return "Cash Withdrawal"


    # --------------------------------------------------------
    # Personal / P2P transfers
    # --------------------------------------------------------

    if "UPI-" in text:

        if (
            "@" in text
            and "SWIGGY" not in text
            and "ZOMATO" not in text
            and "AMAZON" not in text
            and "ZEPTO" not in text
            and "ZERODHA" not in text
        ):
            return "P2P Transfer"


    # --------------------------------------------------------
    # Check vendor dictionary
    # --------------------------------------------------------

    for vendor, keywords in vendor_keywords.items():

        for keyword in keywords:

            if keyword in text:
                return vendor


    # --------------------------------------------------------
    # If no vendor is found
    # --------------------------------------------------------

    return "Uncategorised"


# ------------------------------------------------------------
# 4. Apply vendor extraction
# ------------------------------------------------------------

df_clean["vendor_clean"] = (
    df_clean["Description"]
    .apply(extract_vendor)
)


# ------------------------------------------------------------
# 5. Number of unique vendors
# ------------------------------------------------------------

unique_vendors = df_clean["vendor_clean"].nunique()


print("\nNumber of unique vendors:")
print(unique_vendors)


# ------------------------------------------------------------
# 6. Top 10 vendors
# ------------------------------------------------------------

print("\nTop 10 vendors:")

print(
    df_clean["vendor_clean"]
    .value_counts()
    .head(10)
)


# ------------------------------------------------------------
# 7. Check uncategorised descriptions
# ------------------------------------------------------------

uncategorised = df_clean[
    df_clean["vendor_clean"] == "Uncategorised"
]


print("\nUncategorised transactions:")
print(len(uncategorised))


if len(uncategorised) > 0:

    print("\nDescriptions that were not mapped:")

    for description in uncategorised["Description"].unique():
        print(description)


# ------------------------------------------------------------
# 8. Display sample results
# ------------------------------------------------------------

print("\nSample vendor extraction:")

print(
    df_clean[
        [
            "Description",
            "vendor_clean"
        ]
    ].head(20)
)


# ------------------------------------------------------------
# 9. Feature 2 summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FEATURE 2 COMPLETED")
print("=" * 60)

print(
    f"Unique canonical vendors : {unique_vendors}"
)

print(
    f"Uncategorised transactions: {len(uncategorised)}"
)

print("=" * 60)

FEATURE 2 - VENDOR EXTRACTOR

Unique transaction descriptions:
AMAZON SELLER SVCS
UPI-AMAN-8934@OKAXIS
BHIM ZEPTO
UPI-UBER-2426@HDFCBANK
POS SWIGGY BANGALORE
ANI Technologies
POS SWIGGY-RESTAURANT
POS UBER BANGALORE
TWC INDIA
UPI-STARBUCKS@AXIS
POS BLINKIT
ZOMATO MEDIA P L
BHIM SWIGGY
OLA CABS
UPI-DMART@OKAXIS
POS ZEPTO BANGALORE
UPI-MYNTRA@HDFCBANK
BANGALORE ELEC SUPPLY
Amazon Pay India
UPI-PRIYA-2221@OKAXIS
INSTAMART BANGALORE
UPI-ZOMATO-LIMITED@PAYTM
UPI-SWIGGY-7959@HDFCBANK
UPI-SWIGGY-INSTAMART@OKAXIS
POS OLA-PRIME
POS EMPIRE RESTAURANT
SWIGGY-INSTAMART
POS NYKAA BANGALORE
ROPPEN TRANSPORTATION
UPI-NYKAA@AXIS
UPI-SWIGGY4619@OKAXIS
UPI-FLIPKART@HDFCBANK
IMPS ZERODHA-COIN
UPI-THIRDWAVE@OKAXIS
UPI-SWIGGY-1309@HDFCBANK
NEFT-TECHCRUSH LABS-SALARY MAY24
BHIM-BLINKIT
UPI-ZEPTO-4547@HDFCBANK
RAPIDO BIKE TAXI
UPI-RESTAURANT-6377@PAYTM
Swiggy*Order
COFFEE DAY GLOBAL
FLIPKART INDIA
UPI-CCD@HDFCBANK
UPI-VIKAS-5416@OKAXIS
UPI-BOOKMYSHOW@HDFCBANK
UPI-HOTSTAR@AXIS
UPI-ZOMATO6436@OKICICI
NETFLIX.C

In [11]:
# ============================================================
# FEATURE 3 - CATEGORY TAGGER
# ============================================================

print("=" * 60)
print("FEATURE 3 - CATEGORY TAGGER")
print("=" * 60)


# ------------------------------------------------------------
# 1. Vendor to category mapping
# ------------------------------------------------------------

category_mapping = {

    # Food Delivery
    "Swiggy": "Food Delivery",
    "Zomato": "Food Delivery",

    # Quick Commerce
    "Zepto": "Quick Commerce",
    "Blinkit": "Quick Commerce",
    "BigBasket": "Quick Commerce",

    # E-commerce
    "Amazon": "E-commerce",
    "Myntra": "E-commerce",
    "Flipkart": "E-commerce",
    "Ajio": "E-commerce",

    # Transport
    "Uber": "Transport",
    "Ola": "Transport",
    "Rapido": "Transport",

    # Cafe
    "Cafe Coffee Day": "Cafe",
    "Starbucks": "Cafe",

    # Subscriptions
    "Netflix": "Subscriptions",
    "Spotify": "Subscriptions",
    "YouTube": "Subscriptions",
    "Amazon Prime": "Subscriptions",
    "Hotstar": "Subscriptions",

    # Entertainment
    "BookMyShow": "Entertainment",
    "PVR": "Entertainment",
    "INOX": "Entertainment",

    # Investments
    "Zerodha": "Investments",

    # Fuel
    "HP": "Fuel",
    "Indian Oil": "Fuel",
    "Bharat Petroleum": "Fuel",

    # Personal Transfer
    "P2P Transfer": "Personal Transfer",

    # Cash Withdrawal
    "Cash Withdrawal": "Cash Withdrawal"
}


# ------------------------------------------------------------
# 2. Map vendors to categories
# ------------------------------------------------------------

df_clean["category"] = (
    df_clean["vendor_clean"]
    .map(category_mapping)
)


# ------------------------------------------------------------
# 3. Handle vendors that were not mapped
# ------------------------------------------------------------

df_clean["category"] = (
    df_clean["category"]
    .fillna("Uncategorised")
)


# ------------------------------------------------------------
# 4. Display category counts
# ------------------------------------------------------------

print("\nTransaction count by category:")
print()

print(
    df_clean["category"]
    .value_counts()
)


# ------------------------------------------------------------
# 5. Calculate category spending
# ------------------------------------------------------------

debit_data = df_clean[
    df_clean["type"] == "debit"
].copy()


category_spend = (
    debit_data
    .groupby("category")["amount"]
    .sum()
    .sort_values(ascending=False)
)


# ------------------------------------------------------------
# 6. Display category spending
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("SPENDING BY CATEGORY")
print("=" * 60)

for category, amount in category_spend.items():

    print(
        f"{category:<20} Rs. {amount:>12,.2f}"
    )


# ------------------------------------------------------------
# 7. Check all categories
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CATEGORY CHECK")
print("=" * 60)

print(
    "Total categories found:",
    df_clean["category"].nunique()
)

print("\nCategories:")

for category in sorted(
    df_clean["category"].unique()
):
    print("-", category)


# ------------------------------------------------------------
# 8. Check uncategorised transactions
# ------------------------------------------------------------

uncategorised = df_clean[
    df_clean["category"] == "Uncategorised"
]


print("\n" + "=" * 60)
print("UNCATEGORISED CHECK")
print("=" * 60)

print(
    "Uncategorised transactions:",
    len(uncategorised)
)

if len(uncategorised) > 0:

    print("\nUncategorised vendors:")

    print(
        uncategorised["vendor_clean"]
        .value_counts()
    )


# ------------------------------------------------------------
# 9. Final Feature 3 output
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FEATURE 3 COMPLETED")
print("=" * 60)

print(
    "Every transaction now has a category."
)

print(
    "DataFrame shape:",
    df_clean.shape
)

print("=" * 60)

FEATURE 3 - CATEGORY TAGGER

Transaction count by category:

category
Food Delivery        44
Uncategorised        33
Personal Transfer    28
Transport            15
Quick Commerce        7
E-commerce            6
Investments           4
Cash Withdrawal       3
Subscriptions         1
Fuel                  1
Cafe                  1
Name: count, dtype: int64

SPENDING BY CATEGORY
Investments          Rs.    60,000.00
Uncategorised        Rs.    58,829.00
Personal Transfer    Rs.    38,070.00
Food Delivery        Rs.    19,617.00
E-commerce           Rs.    16,008.00
Transport            Rs.     4,962.00
Quick Commerce       Rs.     3,094.00
Cash Withdrawal      Rs.     3,000.00
Fuel                 Rs.     1,087.00
Cafe                 Rs.       433.00
Subscriptions        Rs.       210.00

CATEGORY CHECK
Total categories found: 11

Categories:
- Cafe
- Cash Withdrawal
- E-commerce
- Food Delivery
- Fuel
- Investments
- Personal Transfer
- Quick Commerce
- Subscriptions
- Transport
- Un

In [12]:
# ============================================================
# FEATURE 4 - SPENDING OVERVIEW
# ============================================================

print("=" * 65)
print("FEATURE 4 - SPENDING OVERVIEW")
print("=" * 65)


# ------------------------------------------------------------
# 1. Separate credits and debits
# ------------------------------------------------------------

credit_data = df_clean[
    df_clean["type"] == "credit"
].copy()

debit_data = df_clean[
    df_clean["type"] == "debit"
].copy()


# ------------------------------------------------------------
# 2. Calculate total credits
# ------------------------------------------------------------

total_credits = credit_data["amount"].sum()


# ------------------------------------------------------------
# 3. Calculate total debits
# ------------------------------------------------------------

total_debits = debit_data["amount"].sum()


# ------------------------------------------------------------
# 4. Calculate net savings
# ------------------------------------------------------------

net_savings = total_credits - total_debits


# ------------------------------------------------------------
# 5. Calculate savings rate
# ------------------------------------------------------------

if total_credits != 0:
    savings_rate = (
        (total_credits - total_debits)
        / total_credits
    ) * 100
else:
    savings_rate = 0


# ------------------------------------------------------------
# 6. Total transaction count
# ------------------------------------------------------------

transaction_count = len(df_clean)


# ------------------------------------------------------------
# 7. Unique vendor count
# ------------------------------------------------------------

unique_vendor_count = (
    df_clean["vendor_clean"].nunique()
)


# ------------------------------------------------------------
# 8. Top 5 categories by spending
# ------------------------------------------------------------

top_categories = (
    debit_data
    .groupby("category")["amount"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)


# ------------------------------------------------------------
# 9. Top 5 vendors by spending
# ------------------------------------------------------------

top_vendors = (
    debit_data
    .groupby("vendor_clean")["amount"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)


# ------------------------------------------------------------
# 10. Calculate category percentages
# ------------------------------------------------------------

category_percentage = (
    top_categories / total_debits
) * 100


# ------------------------------------------------------------
# 11. Print Executive Summary
# ------------------------------------------------------------

print("\nEXECUTIVE SUMMARY")
print("-" * 65)

print(
    f"Total credits       : Rs. {total_credits:,.2f}"
)

print(
    f"Total debits        : Rs. {total_debits:,.2f}"
)

print(
    f"Net savings         : Rs. {net_savings:,.2f}"
)

print(
    f"Savings rate        : {savings_rate:.2f}%"
)

print(
    f"Transactions        : {transaction_count}"
)

print(
    f"Unique vendors      : {unique_vendor_count}"
)


# ------------------------------------------------------------
# 12. Spending status
# ------------------------------------------------------------

if net_savings > 0:
    spending_status = "SAVING"
elif net_savings < 0:
    spending_status = "OVERSPENDING"
else:
    spending_status = "BREAK-EVEN"


print(
    f"Financial status    : {spending_status}"
)


# ------------------------------------------------------------
# 13. Top 5 Categories
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("TOP 5 CATEGORIES")
print("=" * 65)

for category, amount in top_categories.items():

    percentage = category_percentage[category]

    print(
        f"{category:<20} "
        f"{percentage:>6.2f}%   "
        f"Rs. {amount:>12,.2f}"
    )


# ------------------------------------------------------------
# 14. Top 5 Vendors
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("TOP 5 VENDORS")
print("=" * 65)

for vendor, amount in top_vendors.items():

    order_count = len(
        debit_data[
            debit_data["vendor_clean"] == vendor
        ]
    )

    print(
        f"{vendor:<20} "
        f"Rs. {amount:>12,.2f} "
        f"({order_count} transactions)"
    )


# ------------------------------------------------------------
# 15. Feature 4 summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FEATURE 4 COMPLETED")
print("=" * 65)

print(
    f"Total Credits : Rs. {total_credits:,.2f}"
)

print(
    f"Total Debits  : Rs. {total_debits:,.2f}"
)

print(
    f"Net Savings   : Rs. {net_savings:,.2f}"
)

print(
    f"Savings Rate  : {savings_rate:.2f}%"
)

print(
    f"Transactions  : {transaction_count}"
)

print("=" * 65)

FEATURE 4 - SPENDING OVERVIEW

EXECUTIVE SUMMARY
-----------------------------------------------------------------
Total credits       : Rs. 170,094.00
Total debits        : Rs. 205,310.00
Net savings         : Rs. -35,216.00
Savings rate        : -20.70%
Transactions        : 143
Unique vendors      : 16
Financial status    : OVERSPENDING

TOP 5 CATEGORIES
Investments           29.22%   Rs.    60,000.00
Uncategorised         28.65%   Rs.    58,829.00
Personal Transfer     18.54%   Rs.    38,070.00
Food Delivery          9.55%   Rs.    19,617.00
E-commerce             7.80%   Rs.    16,008.00

TOP 5 VENDORS
Zerodha              Rs.    60,000.00 (4 transactions)
Uncategorised        Rs.    58,829.00 (31 transactions)
P2P Transfer         Rs.    38,070.00 (28 transactions)
Swiggy               Rs.    12,638.00 (28 transactions)
Flipkart             Rs.     8,458.00 (3 transactions)

FEATURE 4 COMPLETED
Total Credits : Rs. 170,094.00
Total Debits  : Rs. 205,310.00
Net Savings   : Rs. -35,

In [13]:
# ============================================================
# FEATURE 5 - MONTHLY TREND ANALYSIS
# ============================================================

import numpy as np

print("=" * 65)
print("FEATURE 5 - MONTHLY TREND ANALYSIS")
print("=" * 65)


# ------------------------------------------------------------
# 1. Create month column
# ------------------------------------------------------------

df_clean["month"] = df_clean["date"].dt.month


# ------------------------------------------------------------
# 2. Create month names
# ------------------------------------------------------------

month_names = {
    1: "Jan",
    2: "Feb",
    3: "Mar",
    4: "Apr",
    5: "May",
    6: "Jun"
}

df_clean["month_name"] = (
    df_clean["month"].map(month_names)
)


# ------------------------------------------------------------
# 3. Select debit transactions
# ------------------------------------------------------------

monthly_data = df_clean[
    df_clean["type"] == "debit"
].copy()


# ------------------------------------------------------------
# 4. Create category × month matrix
# ------------------------------------------------------------

month_pivot = monthly_data.pivot_table(
    values="amount",
    index="category",
    columns="month",
    aggfunc="sum",
    fill_value=0
)


# ------------------------------------------------------------
# 5. Make sure all six months are present
# ------------------------------------------------------------

for month in range(1, 7):

    if month not in month_pivot.columns:
        month_pivot[month] = 0


# ------------------------------------------------------------
# 6. Arrange months in correct order
# ------------------------------------------------------------

month_pivot = month_pivot[
    [1, 2, 3, 4, 5, 6]
]


# ------------------------------------------------------------
# 7. Rename month columns
# ------------------------------------------------------------

month_pivot.columns = [
    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "May",
    "Jun"
]


# ------------------------------------------------------------
# 8. Display monthly spending matrix
# ------------------------------------------------------------

print("\nCATEGORY × MONTH SPENDING")
print("-" * 65)

print(
    month_pivot.round(2)
)


# ------------------------------------------------------------
# 9. Calculate total spending for each month
# ------------------------------------------------------------

monthly_totals = (
    monthly_data
    .groupby("month")["amount"]
    .sum()
)


# ------------------------------------------------------------
# 10. Display monthly totals
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("TOTAL SPENDING BY MONTH")
print("=" * 65)

for month in range(1, 7):

    amount = monthly_totals.get(
        month,
        0
    )

    print(
        f"{month_names[month]:<8} "
        f"Rs. {amount:>12,.2f}"
    )


# ------------------------------------------------------------
# 11. Calculate first-to-last month growth
# ------------------------------------------------------------

trend_data = pd.DataFrame({
    "January": month_pivot["Jan"],
    "June": month_pivot["Jun"]
})


# ------------------------------------------------------------
# 12. Calculate percentage change
# ------------------------------------------------------------

trend_data["growth_percent"] = np.where(
    trend_data["January"] != 0,
    (
        (
            trend_data["June"]
            - trend_data["January"]
        )
        / trend_data["January"]
    ) * 100,
    np.nan
)


# ------------------------------------------------------------
# 13. Sort categories by growth
# ------------------------------------------------------------

trend_data = trend_data.sort_values(
    "growth_percent",
    ascending=False
)


# ------------------------------------------------------------
# 14. Find biggest growth
# ------------------------------------------------------------

valid_growth = trend_data.dropna(
    subset=["growth_percent"]
)

if len(valid_growth) > 0:

    biggest_growth_category = (
        valid_growth["growth_percent"]
        .idxmax()
    )

    biggest_growth_value = (
        valid_growth.loc[
            biggest_growth_category,
            "growth_percent"
        ]
    )

else:

    biggest_growth_category = "N/A"
    biggest_growth_value = 0


# ------------------------------------------------------------
# 15. Find biggest decline
# ------------------------------------------------------------

if len(valid_growth) > 0:

    biggest_decline_category = (
        valid_growth["growth_percent"]
        .idxmin()
    )

    biggest_decline_value = (
        valid_growth.loc[
            biggest_decline_category,
            "growth_percent"
        ]
    )

else:

    biggest_decline_category = "N/A"
    biggest_decline_value = 0


# ------------------------------------------------------------
# 16. Display trend analysis
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("TREND ANALYSIS")
print("=" * 65)

print(
    f"Biggest growth  : "
    f"{biggest_growth_category} "
    f"({biggest_growth_value:+.2f}%)"
)

print(
    f"Biggest decline : "
    f"{biggest_decline_category} "
    f"({biggest_decline_value:+.2f}%)"
)


# ------------------------------------------------------------
# 17. Display category growth table
# ------------------------------------------------------------

print("\nCATEGORY GROWTH: JANUARY → JUNE")
print("-" * 65)

for category, row in trend_data.iterrows():

    january = row["January"]
    june = row["June"]
    growth = row["growth_percent"]

    if pd.isna(growth):
        growth_text = "N/A"
    else:
        growth_text = f"{growth:+.2f}%"

    print(
        f"{category:<20} "
        f"Jan: Rs.{january:>10,.2f}  "
        f"Jun: Rs.{june:>10,.2f}  "
        f"{growth_text}"
    )


# ------------------------------------------------------------
# 18. Feature 5 completed
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FEATURE 5 COMPLETED")
print("=" * 65)

print(
    "Monthly category spending matrix created."
)

print(
    "January to June trend calculated."
)

print(
    f"Biggest growth : {biggest_growth_category}"
)

print(
    f"Biggest decline: {biggest_decline_category}"
)

print("=" * 65)

FEATURE 5 - MONTHLY TREND ANALYSIS

CATEGORY × MONTH SPENDING
-----------------------------------------------------------------
                      Jan     Feb      Mar     Apr      May     Jun
category                                                           
Cafe                  0.0     0.0      0.0     0.0      0.0     0.0
Cash Withdrawal       0.0     0.0      0.0     0.0      0.0     0.0
E-commerce         2462.0   996.0   2147.0  3311.0      0.0  3000.0
Food Delivery      1180.0  3454.0   2862.0   632.0   1159.0  1636.0
Fuel                  0.0     0.0      0.0     0.0      0.0     0.0
Investments           0.0     0.0      0.0     0.0      0.0     0.0
Personal Transfer  5155.0   355.0   1714.0   790.0    166.0  1860.0
Quick Commerce      825.0   697.0      0.0     0.0    502.0     0.0
Subscriptions         0.0     0.0      0.0     0.0      0.0     0.0
Transport           678.0   621.0    439.0     0.0    431.0   134.0
Uncategorised      4672.0   267.0  19416.0   513.0  1875

In [14]:
# ============================================================
# FEATURE 6 - TIME-OF-DAY PATTERNS
# ============================================================

import numpy as np

print("=" * 65)
print("FEATURE 6 - TIME-OF-DAY PATTERNS")
print("=" * 65)


# ------------------------------------------------------------
# 1. Extract hour from Time column
# ------------------------------------------------------------

df_clean["hour"] = (
    df_clean["Time"]
    .astype(str)
    .str[:2]
    .astype(int)
)


# ------------------------------------------------------------
# 2. Select debit transactions
# ------------------------------------------------------------

time_data = df_clean[
    df_clean["type"] == "debit"
].copy()


# ------------------------------------------------------------
# 3. Create category × hour spending matrix
# ------------------------------------------------------------

time_matrix = time_data.pivot_table(
    values="amount",
    index="category",
    columns="hour",
    aggfunc="sum",
    fill_value=0
)


# ------------------------------------------------------------
# 4. Make sure all 24 hours are present
# ------------------------------------------------------------

for hour in range(24):

    if hour not in time_matrix.columns:
        time_matrix[hour] = 0


# ------------------------------------------------------------
# 5. Arrange hours from 00 to 23
# ------------------------------------------------------------

time_matrix = time_matrix[
    list(range(24))
]


# ------------------------------------------------------------
# 6. Display spending matrix
# ------------------------------------------------------------

print("\nCATEGORY × HOUR SPENDING MATRIX")
print("-" * 65)

print(
    time_matrix.round(2)
)


# ------------------------------------------------------------
# 7. Find spending peak for each category
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PEAK SPENDING HOUR BY CATEGORY")
print("=" * 65)

for category in time_matrix.index:

    peak_hour = (
        time_matrix.loc[category]
        .idxmax()
    )

    peak_amount = (
        time_matrix.loc[category, peak_hour]
    )

    print(
        f"{category:<20} "
        f"{int(peak_hour):02d}:00  "
        f"Rs. {peak_amount:,.2f}"
    )


# ------------------------------------------------------------
# 8. Food Delivery late-night analysis
# ------------------------------------------------------------

food_delivery = time_data[
    time_data["category"] == "Food Delivery"
].copy()


# ------------------------------------------------------------
# 9. Select transactions between 21:00 and 02:00
# ------------------------------------------------------------

late_night_food = food_delivery[
    (food_delivery["hour"] >= 21)
    | (food_delivery["hour"] <= 2)
]


# ------------------------------------------------------------
# 10. Calculate late-night percentage
# ------------------------------------------------------------

total_food_transactions = len(
    food_delivery
)

late_night_transactions = len(
    late_night_food
)


if total_food_transactions > 0:

    late_night_percentage = (
        late_night_transactions
        / total_food_transactions
    ) * 100

else:

    late_night_percentage = 0


# ------------------------------------------------------------
# 11. Food Delivery spending percentage
# ------------------------------------------------------------

total_food_spend = food_delivery[
    "amount"
].sum()

late_night_food_spend = late_night_food[
    "amount"
].sum()


if total_food_spend > 0:

    late_night_spend_percentage = (
        late_night_food_spend
        / total_food_spend
    ) * 100

else:

    late_night_spend_percentage = 0


# ------------------------------------------------------------
# 12. Cafe morning analysis
# ------------------------------------------------------------

cafe_data = time_data[
    time_data["category"] == "Cafe"
].copy()


morning_cafe = cafe_data[
    (cafe_data["hour"] >= 8)
    & (cafe_data["hour"] <= 11)
]


total_cafe_transactions = len(
    cafe_data
)

morning_cafe_transactions = len(
    morning_cafe
)


if total_cafe_transactions > 0:

    morning_cafe_percentage = (
        morning_cafe_transactions
        / total_cafe_transactions
    ) * 100

else:

    morning_cafe_percentage = 0


# ------------------------------------------------------------
# 13. Print Food Delivery result
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FOOD DELIVERY - LATE NIGHT ANALYSIS")
print("=" * 65)

print(
    f"Total Food Delivery transactions : "
    f"{total_food_transactions}"
)

print(
    f"Transactions after 21:00         : "
    f"{late_night_transactions}"
)

print(
    f"Late-night percentage            : "
    f"{late_night_percentage:.2f}%"
)

print(
    f"Late-night Food spend            : "
    f"Rs. {late_night_food_spend:,.2f}"
)

print(
    f"Late-night spend percentage      : "
    f"{late_night_spend_percentage:.2f}%"
)


# ------------------------------------------------------------
# 14. Print Cafe result
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("CAFE - MORNING ANALYSIS")
print("=" * 65)

print(
    f"Total Cafe transactions : "
    f"{total_cafe_transactions}"
)

print(
    f"08:00 - 11:00 orders    : "
    f"{morning_cafe_transactions}"
)

print(
    f"Morning percentage      : "
    f"{morning_cafe_percentage:.2f}%"
)


# ------------------------------------------------------------
# 15. Simple ASCII hourly spending display
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("HOURLY SPENDING")
print("=" * 65)

hourly_total = (
    time_data
    .groupby("hour")["amount"]
    .sum()
)


max_hourly_spend = hourly_total.max()


for hour in range(24):

    amount = hourly_total.get(
        hour,
        0
    )

    if max_hourly_spend > 0:

        bar_length = int(
            (amount / max_hourly_spend) * 30
        )

    else:

        bar_length = 0

    bar = "#" * bar_length

    print(
        f"{hour:02d}:00 | "
        f"{bar:<30} "
        f"Rs. {amount:,.2f}"
    )


# ------------------------------------------------------------
# 16. Feature 6 summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FEATURE 6 COMPLETED")
print("=" * 65)

print(
    f"Food Delivery late-night orders: "
    f"{late_night_percentage:.2f}%"
)

print(
    f"Cafe orders from 08:00-11:00: "
    f"{morning_cafe_percentage:.2f}%"
)

print("=" * 65)

FEATURE 6 - TIME-OF-DAY PATTERNS

CATEGORY × HOUR SPENDING MATRIX
-----------------------------------------------------------------
hour                  0      1      2       3        4       5       6   \
category                                                                  
Cafe                 0.0    0.0    0.0     0.0      0.0     0.0     0.0   
Cash Withdrawal      0.0    0.0    0.0     0.0      0.0     0.0     0.0   
E-commerce           0.0    0.0    0.0  2462.0      0.0     0.0     0.0   
Food Delivery      394.0    0.0  318.0     0.0    413.0   537.0     0.0   
Fuel                 0.0    0.0    0.0     0.0      0.0     0.0     0.0   
Investments          0.0    0.0    0.0     0.0  15000.0     0.0     0.0   
Personal Transfer  562.0  842.0    0.0     0.0      0.0  4496.0     0.0   
Quick Commerce       0.0    0.0  697.0     0.0      0.0     0.0     0.0   
Subscriptions        0.0    0.0    0.0     0.0      0.0     0.0     0.0   
Transport          431.0    0.0    0.0     

In [15]:
# ============================================================
# FEATURE 7 - ANOMALY DETECTION
# ============================================================

print("=" * 65)
print("FEATURE 7 - ANOMALY DETECTION")
print("=" * 65)


# ------------------------------------------------------------
# 1. Select debit transactions
# ------------------------------------------------------------

anomaly_data = df_clean[
    df_clean["type"] == "debit"
].copy()


# ------------------------------------------------------------
# 2. Calculate category-wise mean
# ------------------------------------------------------------

category_mean = (
    anomaly_data
    .groupby("category")["amount"]
    .transform("mean")
)


# ------------------------------------------------------------
# 3. Calculate category-wise standard deviation
# ------------------------------------------------------------

category_std = (
    anomaly_data
    .groupby("category")["amount"]
    .transform("std")
)


# ------------------------------------------------------------
# 4. Calculate Z-score
# ------------------------------------------------------------

anomaly_data["z_score"] = (
    anomaly_data["amount"] - category_mean
) / category_std


# ------------------------------------------------------------
# 5. Find anomalous transactions
# ------------------------------------------------------------

anomalies = anomaly_data[
    anomaly_data["z_score"] > 2
].copy()


# ------------------------------------------------------------
# 6. Sort anomalies by z-score
# ------------------------------------------------------------

anomalies = anomalies.sort_values(
    "z_score",
    ascending=False
)


# ------------------------------------------------------------
# 7. Display anomaly count
# ------------------------------------------------------------

print("\nTotal debit transactions:")
print(len(anomaly_data))

print("\nTotal anomalies detected:")
print(len(anomalies))


# ------------------------------------------------------------
# 8. Display top 5 anomalies
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("TOP 5 ANOMALOUS TRANSACTIONS")
print("=" * 65)

top_anomalies = anomalies.head(5)

for index, row in top_anomalies.iterrows():

    transaction_date = row["date"].strftime(
        "%d-%b-%Y"
    )

    print(
        f"\nDate     : {transaction_date}"
    )

    print(
        f"Vendor   : {row['vendor_clean']}"
    )

    print(
        f"Category : {row['category']}"
    )

    print(
        f"Amount   : Rs. {row['amount']:,.2f}"
    )

    print(
        f"Z-score  : {row['z_score']:.2f}"
    )


# ------------------------------------------------------------
# 9. Display all anomalies as a table
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("ALL ANOMALIES")
print("=" * 65)

if len(anomalies) > 0:

    print(
        anomalies[
            [
                "date",
                "vendor_clean",
                "category",
                "amount",
                "z_score"
            ]
        ].to_string(
            index=False
        )
    )

else:

    print("No anomalies detected.")


# ------------------------------------------------------------
# 10. Count anomalies by category
# ------------------------------------------------------------

anomaly_category_counts = (
    anomalies["category"]
    .value_counts()
)


print("\n" + "=" * 65)
print("ANOMALIES BY CATEGORY")
print("=" * 65)

print(
    anomaly_category_counts
)


# ------------------------------------------------------------
# 11. Feature 7 summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FEATURE 7 COMPLETED")
print("=" * 65)

print(
    f"Anomalies detected: {len(anomalies)}"
)

if len(anomalies) > 0:

    highest_anomaly = anomalies.iloc[0]

    print(
        f"Highest anomaly : "
        f"{highest_anomaly['vendor_clean']}"
    )

    print(
        f"Amount          : "
        f"Rs. {highest_anomaly['amount']:,.2f}"
    )

    print(
        f"Z-score         : "
        f"{highest_anomaly['z_score']:.2f}"
    )

print("=" * 65)

FEATURE 7 - ANOMALY DETECTION

Total debit transactions:
141

Total anomalies detected:
3

TOP 5 ANOMALOUS TRANSACTIONS

Date     : 01-Sep-2024
Vendor   : P2P Transfer
Category : Personal Transfer
Amount   : Rs. 10,745.00
Z-score  : 4.52

Date     : 05-May-2024
Vendor   : Uncategorised
Category : Uncategorised
Amount   : Rs. 18,000.00
Z-score  : 3.70

Date     : 06-Mar-2024
Vendor   : Uncategorised
Category : Uncategorised
Amount   : Rs. 18,000.00
Z-score  : 3.70

ALL ANOMALIES
      date  vendor_clean          category  amount  z_score
2024-09-01  P2P Transfer Personal Transfer 10745.0 4.521389
2024-05-05 Uncategorised     Uncategorised 18000.0 3.703112
2024-03-06 Uncategorised     Uncategorised 18000.0 3.703112

ANOMALIES BY CATEGORY
category
Uncategorised        2
Personal Transfer    1
Name: count, dtype: int64

FEATURE 7 COMPLETED
Anomalies detected: 3
Highest anomaly : P2P Transfer
Amount          : Rs. 10,745.00
Z-score         : 4.52


In [16]:
# ============================================================
# FEATURE 8 - SPENDING ARCHETYPE DETECTION
# ============================================================

print("=" * 65)
print("FEATURE 8 - SPENDING ARCHETYPE DETECTION")
print("=" * 65)


# ------------------------------------------------------------
# 1. Use debit transactions only
# ------------------------------------------------------------

archetype_data = df_clean[
    df_clean["type"] == "debit"
].copy()


# ------------------------------------------------------------
# 2. Calculate total spending
# ------------------------------------------------------------

total_spending = archetype_data["amount"].sum()


# ------------------------------------------------------------
# 3. Calculate spending by category
# ------------------------------------------------------------

category_spending = (
    archetype_data
    .groupby("category")["amount"]
    .sum()
)


# ------------------------------------------------------------
# 4. Calculate spending shares
# ------------------------------------------------------------

if total_spending > 0:

    category_share = (
        category_spending
        / total_spending
    )

else:

    category_share = category_spending * 0


# ------------------------------------------------------------
# 5. Get important category shares
# ------------------------------------------------------------

food_share = (
    category_share.get(
        "Food Delivery",
        0
    )
    +
    category_share.get(
        "Restaurants",
        0
    )
    +
    category_share.get(
        "Cafe",
        0
    )
)


shopping_share = (
    category_share.get(
        "E-commerce",
        0
    )
    +
    category_share.get(
        "Quick Commerce",
        0
    )
)


investment_share = category_share.get(
    "Investments",
    0
)


transport_share = category_share.get(
    "Transport",
    0
)


# ------------------------------------------------------------
# 6. Convert shares into percentages
# ------------------------------------------------------------

food_percentage = food_share * 100

shopping_percentage = shopping_share * 100

investment_percentage = investment_share * 100

transport_percentage = transport_share * 100


# ------------------------------------------------------------
# 7. Determine dominant category
# ------------------------------------------------------------

archetype_scores = {

    "Foodie": food_share,

    "Shopper": shopping_share,

    "Investor": investment_share,

    "Commuter": transport_share
}


dominant_archetype = max(
    archetype_scores,
    key=archetype_scores.get
)


dominant_share = archetype_scores[
    dominant_archetype
]


# ------------------------------------------------------------
# 8. Assign spending archetype
# ------------------------------------------------------------

# If no single behaviour is strong enough,
# classify the user as Balanced.

if dominant_share >= 0.30:

    archetype = dominant_archetype

else:

    archetype = "Balanced"


# ------------------------------------------------------------
# 9. Display spending shares
# ------------------------------------------------------------

print("\nSPENDING SHARES")
print("-" * 65)

print(
    f"Food-related spending : "
    f"{food_percentage:.2f}%"
)

print(
    f"Shopping spending     : "
    f"{shopping_percentage:.2f}%"
)

print(
    f"Investment spending   : "
    f"{investment_percentage:.2f}%"
)

print(
    f"Transport spending    : "
    f"{transport_percentage:.2f}%"
)


# ------------------------------------------------------------
# 10. Display all category shares
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("ALL CATEGORY SPENDING SHARES")
print("=" * 65)

sorted_shares = (
    category_share
    .sort_values(
        ascending=False
    )
)

for category, share in sorted_shares.items():

    print(
        f"{category:<20} "
        f"{share * 100:>7.2f}%"
    )


# ------------------------------------------------------------
# 11. Display archetype
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("SPENDING ARCHETYPE")
print("=" * 65)

print(
    f"Archetype : {archetype}"
)

print(
    f"Dominant share : "
    f"{dominant_share * 100:.2f}%"
)


# ------------------------------------------------------------
# 12. Give a simple explanation
# ------------------------------------------------------------

if archetype == "Foodie":

    explanation = (
        "Food-related spending forms the largest "
        "significant share of total spending."
    )

elif archetype == "Shopper":

    explanation = (
        "Shopping and e-commerce form the largest "
        "significant share of total spending."
    )

elif archetype == "Investor":

    explanation = (
        "Investment-related spending forms the "
        "largest significant share of spending."
    )

elif archetype == "Commuter":

    explanation = (
        "Transport-related spending forms the "
        "largest significant share of spending."
    )

else:

    explanation = (
        "No single spending behaviour dominates strongly, "
        "so the user is classified as Balanced."
    )


print("\nExplanation:")
print(explanation)


# ------------------------------------------------------------
# 13. Create final archetype result
# ------------------------------------------------------------

archetype_result = pd.DataFrame({

    "Archetype": [archetype],

    "Food_Share_Percent": [
        food_percentage
    ],

    "Shopping_Share_Percent": [
        shopping_percentage
    ],

    "Investment_Share_Percent": [
        investment_percentage
    ],

    "Transport_Share_Percent": [
        transport_percentage
    ],

    "Dominant_Share_Percent": [
        dominant_share * 100
    ]
})


# ------------------------------------------------------------
# 14. Display final result
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FINAL ARCHETYPE RESULT")
print("=" * 65)

print(
    archetype_result.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 15. Feature 8 completed
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FEATURE 8 COMPLETED")
print("=" * 65)

print(
    f"Final Spending Archetype: {archetype}"
)

print("=" * 65)

FEATURE 8 - SPENDING ARCHETYPE DETECTION

SPENDING SHARES
-----------------------------------------------------------------
Food-related spending : 9.77%
Shopping spending     : 9.30%
Investment spending   : 29.22%
Transport spending    : 2.42%

ALL CATEGORY SPENDING SHARES
Investments            29.22%
Uncategorised          28.65%
Personal Transfer      18.54%
Food Delivery           9.55%
E-commerce              7.80%
Transport               2.42%
Quick Commerce          1.51%
Cash Withdrawal         1.46%
Fuel                    0.53%
Cafe                    0.21%
Subscriptions           0.10%

SPENDING ARCHETYPE
Archetype : Balanced
Dominant share : 29.22%

Explanation:
No single spending behaviour dominates strongly, so the user is classified as Balanced.

FINAL ARCHETYPE RESULT
Archetype  Food_Share_Percent  Shopping_Share_Percent  Investment_Share_Percent  Transport_Share_Percent  Dominant_Share_Percent
 Balanced             9.76572                9.303979                   29.